In [1]:
import pandas as pd
import os
#os.chdir('/content/drive/MyDrive/name/IMP-OIC-Windowing')
from utils.extractframes import FrameExtractor
import graphene
from gpt_ask import run_gpt
main_ds = pd.read_json('TVQAP/tvqa_with_subtitle.json')

prompt_ds = pd.read_csv('TVQAP/OIC_response_gpt4_w_st.csv')

#merge and drop generated duplicate rows
main_ds=pd.merge(main_ds,prompt_ds[['qid','OIC_context']],how='left', on='qid')
main = main_ds.loc[main_ds.astype(str).drop_duplicates().index]
main['OIC_context'] = main['OIC_context'].fillna('_') 
main['subtitle'] = main['subtitle'].fillna('_') 
main = main.loc[main['OIC_context']!= '_']
main

,a1,qid,answer_idx,ts,q,a0,a3,a2,bbox,vid_name,a4,subtitle,OIC_context
0,Raj is weird .,133290,3,"[0.0, 5.4]",Why did Raj tell himself to turn his pelvis wh...,Raj was trying to get away from Penny .,Raj had become excited and did not want Penny ...,Raj likes to give himself odd instructions .,"{'1': [{'img_id': 1, 'top': 14, 'height': 346,...",s01e02_seg02_clip_12,Raj did not like hugging Penny .,UNKNAME : Uh oh . Turn your pelvis .,The scene opens with woman_fd00 has hair_c84c\...
1,Howard told Leonard that he would never beat h...,136932,2,"[12.3, 20.11]",What did Howard tell Leonard after he finished...,Howard told Leonard to go pick up his lunch .,Howard told Leonard to compare their scores .,Howard told Leonard to grab a napking because ...,"{'39': [{'img_id': 39, 'top': 43, 'height': 24...",s01e02_seg02_clip_12,Howard told Leonard to copy his dance moves .,UNKNAME : Uh oh . Turn your pelvis .,The scene opens with lamp_9707 on table_6995\n...
2,Leonard told Howard that Howard is n't very go...,133451,3,"[17.71, 25.51]",What did Leonard tell Howard after Howard said...,Leonard told Howard that he really hates that ...,"Leonard told Howard that it was fine , he wins .",Leonard told Howard that Sheldon will beat his...,"{'68': [{'img_id': 68, 'top': 66, 'height': 29...",s01e02_seg02_clip_12,Leonard told Howard that he will beat him .,"UNKNAME : Grab a napkin , homey , you just go...",The scene opens with woman_37b5 has hair_895c\...
3,Raj said that Penny was very happy .,134785,4,"[46.83, 55.84]",Why did Raj say that Penny was upset after Leo...,Raj said that Penny did n't mention Leonard .,Raj said that Penny was only mad at Howard .,Raj said that Penny was very mad at Leonard .,"{'155': [{'img_id': 155, 'top': 23, 'height': ...",s01e02_seg02_clip_12,Raj said that Penny was upset because Penny 's...,"UNKNAME : Grab a napkin , homey , you just go...",The scene opens with man_1a93 wearing glass_c5...
4,Lesley says there was no passion .,131529,0,"[7.67, 16.23]",What does Lesley say there was none of when Le...,Lesley says there was no arousal .,Lesley says the kiss lacked a certain fire .,There was no kiss .,"{'26': [{'img_id': 26, 'top': 20, 'height': 34...",s01e03_seg02_clip_05,Lesley says there was no excitement in the kiss .,Lesley : . no extraneous spittle .,The scene opens with woman_e876 has glass_d9c5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2652,Gee thanks sheldon,130619,0,"[45.91, 53.04]",What did leonard say after sheldon gave him th...,i 'm not a preschooler,I never though this day would come,wow how nice of you,"{'146': [{'img_id': 146, 'top': 90, 'height': ...",s08e14_seg01_clip_02,I did n't see that one coming,Sheldon : That is a sticker of a kitty saying...,The scene opens with man_e746 wearing shirt_34...
2653,Pink,136022,1,"[42.3, 46.53]",What color was Bernadette 's sweater when she ...,Yellow .,White .,Beige .,"{'129': [{'img_id': 129, 'top': 58, 'height': ...",s09e16_seg02_clip_12,Black .,Howard : Mmm . And I 'm not just saying that,The scene opens with glass_9736 on face_7c89\n...
2654,On a chair .,131746,3,"[52.87, 60.43]",Where was Bernadette sitting when Amy and Penn...,A car seat .,On a bench .,A bar stool .,"{'173': [], '167': [{'img_id': 167, 'top': 111...",s09e16_seg02_clip_12,The floor .,Howard : Mmm . And I 'm not just saying that,The scene opens with table_a074 has leg_1a75\n...
2657,she sit down,124258,0,"[29.87, 34.05]",What did Penny do after talking to Raj ?,She left,She answer her phone,she pick up a food,"{'98': [{'img_id': 98, 'top': 42, 'height': 24...",s08e04_seg02_clip_07,She cry,Penny : Great .,The scene opens with woman_4183 has hair_bc07\...


In [2]:
q_ids = main['qid'].unique()
i=0
for qid in q_ids:
  #if 'Seq' in q:
    que = main.query("qid=={}".format(qid))
    question = que['q'].values[0]
    answer_id = main.query("qid=={}".format(qid))["answer_idx"].values[0]
    subtitle = que['subtitle'].values[0]
    prompt = que['OIC_context'].values[0]
    choice_string = ''

    choice_string = "0: {}, 1: {}, 2: {}, 3: {}, 4:{}".format(que['a0'].values[0], que['a1'].values[0],que['a2'].values[0],que['a3'].values[0],que['a4'].values[0])
    
    formatted_question = question+ 'Guess the most likely answer among these options: '+choice_string+' Respond only with a single number between 0 and 4. Do not produce any other output. If enough information is not given, still make a guess to result in one out of the given options.'
    response = run_gpt(prompt+'\n subtitle: '+subtitle, formatted_question)
    #response = run_gpt(prompt, formatted_question)
    main.loc[main['qid'] == qid, 'OIC_answer'] = str(response)
    main.loc[main['qid'] == qid, 'OIC_question'] = formatted_question
    OIC_answer = response
    print(choice_string)
    print(OIC_answer)
   
    if len(OIC_answer)>1:
      main.loc[main['qid'] == qid, 'Match'] = OIC_answer
    else:
      if int(OIC_answer) == int(answer_id):
          main.loc[main['qid'] == qid, 'Match'] = 'Correct'
          print('correct')
      else:
          print('wrong')
          main.loc[main['qid'] == qid, 'Match'] = 'Wrong'
    print('-'*100)
    print('OIC question: {}'.format(formatted_question))
    print('OIC answer: {}'.format(OIC_answer))
    main.to_csv('TVQAP/for_eval/OIC_gpt4_w_st_3.csv')
    i=i+1
    print(i)
    

0: Raj was trying to get away from Penny ., 1: Raj is weird ., 2: Raj likes to give himself odd instructions ., 3: Raj had become excited and did not want Penny to know ., 4:Raj did not like hugging Penny .
3
correct
----------------------------------------------------------------------------------------------------
OIC question: Why did Raj tell himself to turn his pelvis when Penny was giving him a hug ?Guess the most likely answer among these options: 0: Raj was trying to get away from Penny ., 1: Raj is weird ., 2: Raj likes to give himself odd instructions ., 3: Raj had become excited and did not want Penny to know ., 4:Raj did not like hugging Penny . Respond only with a single number between 0 and 4. Do not produce any other output. If enough information is not given, still make a guess.
OIC answer: 3
1
0: Howard told Leonard to go pick up his lunch ., 1: Howard told Leonard that he would never beat his score ., 2: Howard told Leonard to grab a napking because he just got served

KeyboardInterrupt: 